In [1]:
from matplotlib.colors import LinearSegmentedColormap
from datetime import datetime, timedelta
from notebook_utils import calculate_metrics, eval_metrics, timeseries_rel, trim_extremes, catplot_geo
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import geopandas as gpd
import json
import pandas as pd
import numpy as np
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Styling Cell
sns.set_theme(context="notebook", style="darkgrid")

SMALL_SIZE = 18
MEDIUM_SIZE = 24
BIGGER_SIZE = 28

plt.rc('font', size=SMALL_SIZE)          # controls default text sizes
plt.rc('axes', titlesize=SMALL_SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=MEDIUM_SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SMALL_SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SMALL_SIZE)    # legend fontsize
plt.rc('figure', titlesize=BIGGER_SIZE)  # fontsize of the figure title

# To help with visualization, map proper names to the stats
stat_propers = {
    'mae': 'Mean Absolute Error',
    'rmse': 'Root Mean Absolute Error',
    'bias': 'Mean Forecast Bias',
    'corr': 'Correlation Coefficient',
    'skill_score': 'Skill Score'
}

## Import Historical Data

In [4]:
historical = pd.read_csv("../data/monterey_polygon_historical.csv")[["field_id", "crop", "time", "actual_et"]]
historical["time"] = pd.to_datetime(historical["time"])
historical

,field_id,crop,time,actual_et
0,CA_244000,47,2016-01-01,0.499
1,CA_244000,47,2016-01-02,0.641
2,CA_244000,47,2016-01-03,0.728
3,CA_244000,47,2016-01-04,0.520
4,CA_244000,47,2016-01-05,0.565
...,...,...,...,...
2371470,CA_258026,61,2024-12-10,1.129
2371471,CA_258026,61,2024-12-11,1.042
2371472,CA_258026,61,2024-12-12,1.129
2371473,CA_258026,61,2024-12-13,1.216


In [6]:
avgs = pd.read_csv("../data/monterey_polygon_historical_2024_avgs.csv")[["field_id", "crop", "actual_et"]]
avgs

,field_id,crop,actual_et
0,CA_244000,47,2.243913
1,CA_244018,47,1.710027
2,CA_244025,47,2.015713
3,CA_244035,69,2.270014
4,CA_244053,47,1.485169
...,...,...,...
714,CA_257950,47,2.851899
715,CA_257978,47,1.428675
716,CA_257983,47,3.646336
717,CA_258017,47,3.260522


In [9]:
climatology = pd.read_csv("../data/monterey_polygon_historical_climatology.csv")[["field_id", "crop", "doy", "actual_et"]]
climatology

,field_id,crop,doy,actual_et
0,CA_244000,47,1,0.558000
1,CA_244000,47,2,0.595700
2,CA_244000,47,3,0.789900
3,CA_244000,47,4,0.766300
4,CA_244000,47,5,0.659400
...,...,...,...,...
263149,CA_258026,61,362,0.715667
263150,CA_258026,61,363,0.662556
263151,CA_258026,61,364,0.618889
263152,CA_258026,61,365,0.492889


## Import Forecast Data

In [12]:
# Gather current forecast data for the county
forecasts = pd.DataFrame()
files = Path(f"../data/forecasts/fae/monterey/").glob("*.csv")

for file in files:
    parts = str(file.name).split("_")
    data = pd.read_csv(file, low_memory=False)
    data["forecasting_date"] = parts[1].split('.')[0]
    forecasts = pd.concat([data, forecasts], ignore_index=True)

forecasts['forecasting_date'] = pd.to_datetime(forecasts['forecasting_date'])
forecasts['time'] = pd.to_datetime(forecasts['time'])
forecasts

,field_id,crop,time,ff_et,avg_et,med_et,forecasting_date
0,CA_244000,47,2025-05-31,3.612,3.780,4.171,2025-05-31
1,CA_244000,47,2025-06-01,3.956,4.198,4.662,2025-05-31
2,CA_244000,47,2025-06-02,3.642,3.938,4.300,2025-05-31
3,CA_244000,47,2025-06-03,3.393,3.739,4.020,2025-05-31
4,CA_244000,47,2025-06-04,3.362,3.766,3.991,2025-05-31
...,...,...,...,...,...,...,...
5084,CA_258026,61,2025-06-02,7.146,4.300,3.757,2025-05-31
5085,CA_258026,61,2025-06-03,6.845,4.323,3.723,2025-05-31
5086,CA_258026,61,2025-06-04,6.943,4.593,4.345,2025-05-31
5087,CA_258026,61,2025-06-05,7.322,5.066,5.567,2025-05-31
